# Data Analysis Agent (Local Notebook)

Run this notebook locally in Jupyter or VS Code to launch the Web UI without Google Colab.

## Quick Start (Local)
1. Open this notebook locally.
2. Run Cell 2 (setup helpers).
3. Run Cell 3 (start server + clickable localhost URL).
4. Open the printed URL, upload CSV, type a question, and click Analyze Dataset.
5. (Optional) Run Cell 4 to stop the server.

In [ ]:
import json
import os
import sys
import subprocess
import time
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

from IPython.display import HTML, display

PORT = 8000
HOST = "127.0.0.1"
LOCAL_BASE_URL = f"http://{HOST}:{PORT}"

def _mask_secret(value: str | None) -> str:
    if not value:
        return "<not set>"
    if len(value) <= 10:
        return "*" * len(value)
    return f"{value[:6]}...{value[-4:]}"

def _is_project_root(path: Path) -> bool:
    return (path / "app.py").exists() and (path / "static").exists()

def _scan_for_project_root(base: Path, max_depth: int = 5) -> Path | None:
    if not base.exists() or not base.is_dir():
        return None

    base_depth = len(base.parts)
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        current_depth = len(root_path.parts) - base_depth
        if current_depth > max_depth:
            dirs[:] = []
            continue

        if "app.py" in files and (root_path / "static").exists():
            return root_path

    return None

def _find_project_root(start: Path) -> Path:
    hint = os.getenv("PROJECT_ROOT_HINT", "").strip()
    if hint:
        hint_path = Path(hint).expanduser()
        if _is_project_root(hint_path):
            return hint_path

    for candidate in [start, *start.parents]:
        if _is_project_root(candidate):
            return candidate

    common_bases = [Path.cwd(), Path.home(), Path("/tmp")]
    for base in common_bases:
        found = _scan_for_project_root(base, max_depth=6)
        if found is not None:
            return found

    raise FileNotFoundError(
        "Could not locate project root containing app.py and static/.\n"
        "Set PROJECT_ROOT_HINT to your project folder before running this cell."
    )

def _wait_for_server(base_url: str, timeout_seconds: int = 30) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with urlopen(f"{base_url}/health", timeout=2) as response:
                if response.status == 200:
                    return True
        except URLError:
            pass
        time.sleep(1)
    return False

def _get_health(base_url: str):
    try:
        with urlopen(f"{base_url}/health", timeout=3) as response:
            if response.status == 200:
                return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None
    return None

PROJECT_ROOT = _find_project_root(Path.cwd())
SERVER_PROCESS = None

print(f"Project root: {PROJECT_ROOT}")
print("Runtime: Local Jupyter/VS Code")
print(f"Server bind host: {HOST}:{PORT}")
print(f"Base URL: {LOCAL_BASE_URL}")

In [ ]:
# Start Web UI server (Local)
is_running = SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None

if is_running:
    health = _get_health(LOCAL_BASE_URL)
    if health and health.get("agent_initialized") is True:
        print(f"Server is already running and healthy at {LOCAL_BASE_URL}")
    else:
        print("A server is running but the agent is not initialized. Restarting with API key...")
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except Exception:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=5)
        is_running = False

if not is_running:
    env_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if env_key:
        print(f"Detected environment key fingerprint: {_mask_secret(env_key)}")

    if not env_key:
        api_key = getpass("Enter Gemini API key for Web UI: ").strip()
    else:
        api_key = env_key

    if not api_key:
        raise ValueError("Gemini API key is required to start analysis server.")

    print(f"Using key fingerprint for this run: {_mask_secret(api_key)}")

    server_env = os.environ.copy()
    server_env["GEMINI_API_KEY"] = api_key
    server_env["GOOGLE_API_KEY"] = api_key

    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", HOST, "--port", str(PORT)],
        cwd=str(PROJECT_ROOT),
        env=server_env,
    )

    if not _wait_for_server(LOCAL_BASE_URL, timeout_seconds=30):
        raise RuntimeError("Server did not become ready in time. Check notebook output for errors.")

    health = _get_health(LOCAL_BASE_URL)
    if not health or health.get("agent_initialized") is not True:
        status = health.get("status") if isinstance(health, dict) else "unknown"
        detail = health.get("detail") if isinstance(health, dict) else "unknown"
        raise RuntimeError(
            f"Server is reachable but agent initialization failed (status={status}). Detail: {detail}"
        )

BASE_URL = LOCAL_BASE_URL
display(HTML(f'<a href="{BASE_URL}" target="_blank">Open Data Analysis Web UI (Local)</a>'))
print(f"Web UI ready: {BASE_URL}")
print("Local mode: open the URL above in your browser.")
print("Now upload/drop a CSV and click Analyze Dataset.")

In [ ]:
# Optional: stop Web UI server
STOP_SERVER_NOW = False

if STOP_SERVER_NOW and SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=10)
    print("Web UI server stopped.")
elif STOP_SERVER_NOW:
    print("No running Web UI server found.")
else:
    print("Stop skipped. Set STOP_SERVER_NOW = True to stop the server.")